In [ ]:
import os
import cv2


def extract_frames(video_folder, output_folder, frame_skip=60, prefix="img"):
    """
    Extract frames from all videos stored breed-wise.

    Args:
        video_folder (str): Folder containing breed-wise video folders.
        output_folder (str): Directory where extracted frames will be saved.
        frame_skip (int): Save one frame every X frames.
        prefix (str): Prefix for saved image filenames.
    """

    for breed in os.listdir(video_folder):

        # Ignore hidden files such as .DS_Store
        if breed.startswith('.'):
            continue

        video_dir = os.path.join(video_folder, breed)

        # Make sure this is a directory
        if not os.path.isdir(video_dir):
            continue

        # Create corresponding breed folder
        output_dir = os.path.join(output_folder, breed)
        os.makedirs(output_dir, exist_ok=True)

        saved_count = 0

        # Process every video belonging to this breed
        for video_name in os.listdir(video_dir):

            if not video_name.lower().endswith(('.mp4', '.mkv', '.webm')):
                continue

            video_path = os.path.join(video_dir, video_name)

            cap = cv2.VideoCapture(video_path)

            if not cap.isOpened():
                print(f"Could not open video: {video_path}")
                continue

            frame_count = 0

            while True:

                ret, frame = cap.read()

                if not ret:
                    break

                if frame_count % frame_skip == 0:

                    filename = f"{prefix}_frame_{saved_count:04d}.jpg"
                    filepath = os.path.join(output_dir, filename)

                    cv2.imwrite(filepath, frame)

                    saved_count += 1

                frame_count += 1

            cap.release()

        print(
            f"Extraction complete! Saved {saved_count} frames to {output_dir} "
        )

I downloaded videos of cattles in the farms from youtube into the folder named videos grouped by breed name

In [ ]:
video_folder = '/Users/satviksingh/Desktop/creating_dataset_cattle/videos'
output_folder = '/Users/satviksingh/Desktop/creating_dataset_cattle/local_raw_frames'

extract_frames(video_folder, output_folder, frame_skip=60, prefix="img")


# Deduplication of the images

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image
from tqdm import tqdm # for progression bar

def tf_deep_deduplicate(raw_dir, threshold=0.92):
    """
    Uses TensorFlow and MobileNetV2 to extract image features and delete 
    images that are semantically similar (e.g., same pose, slightly tilted head).

    raw_dir: Directory containing subfolders for each breed, each with images.
    threshold: Cosine similarity threshold above which images are considered duplicates.
    """
    print("Loading TensorFlow MobileNetV2 model...")

    model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')

    breed_folders = [f for f in os.listdir(raw_dir) if os.path.isdir(os.path.join(raw_dir, f)) and not f.startswith('.')]

    for breed in breed_folders:
        breed_path = os.path.join(raw_dir, breed)
        images = sorted([f for f in os.listdir(breed_path) if f.endswith(('.jpg', '.jpeg', '.png'))])

        if not images:
            continue
        print(f"\nExtracting features for {breed} ({len(images)} images)...")

        features = []
        valid_images = []

        # 1. BATCH EXTRACTION
        for img_name in tqdm(images):
            img_path = os.path.join(breed_path, img_name)
            try:
                # Loading and preprocessing exactly how MobileNetV2 expects
                img = image.load_img(img_path, target_size=(224, 224))
                x = image.img_to_array(img)
                x = np.expand_dims(x, axis=0)
                x = preprocess_input(x)
                
                #Feature extraction using the model directly
                feat = model(x, training=False).numpy().flatten()
                
                # L2 Normalize the vector so we can use simple dot product later
                feat = feat / np.linalg.norm(feat)
                
                features.append(feat)
                valid_images.append(img_path)
            except Exception as e:
                print(f"Error loading {img_name}: {e}")
                continue

        if not features: continue

        # 2. VECTORIZED MATH: Calculate all similarities instantly using NumPy
        print("Calculating similarities and filtering duplicates...")
        feature_matrix = np.array(features)
        
        # Dot product of normalized vectors gives the cosine similarity
        similarity_matrix = np.dot(feature_matrix, feature_matrix.T)

        to_delete_indices = set()
        deleted_count = 0

        # 3. FILTERING
        for i in range(len(valid_images)):
            if i in to_delete_indices: continue
            
            # Check all subsequent images against the current one
            for j in range(i + 1, len(valid_images)):
                if similarity_matrix[i, j] > threshold:
                    to_delete_indices.add(j)

        # Delete the flagged duplicates
        for idx in to_delete_indices:
            os.remove(valid_images[idx])
            deleted_count += 1

        print(f"-> Removed {deleted_count} similar poses. Kept {len(valid_images) - deleted_count} highly unique images.")


RAW_DIR = "/Users/satviksingh/Desktop/creating_dataset_cattle/local_raw_frames"

tf_deep_deduplicate(RAW_DIR, threshold=0.82)

Loading TensorFlow MobileNetV2 model...


/var/folders/8j/65h65rn171jc_kw4mq2hh4_w0000gn/T/ipykernel_17442/2347379748.py:21: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')



Extracting features for amritmahal (32 images)...


100%|██████████| 32/32 [00:02<00:00, 11.96it/s]


Calculating similarities and filtering duplicates...
-> Removed 19 similar poses. Kept 13 highly unique images.

Extracting features for malvi (18 images)...


...

Calculating similarities and filtering duplicates...
-> Removed 19 similar poses. Kept 20 highly unique images.

Extracting features for red_sindhi (151 images)...


100%|██████████| 151/151 [00:11<00:00, 13.15it/s]


Calculating similarities and filtering duplicates...
-> Removed 68 similar poses. Kept 83 highly unique images.

Extracting features for hariana (636 images)...


100%|██████████| 636/636 [01:00<00:00, 10.50it/s]


Calculating similarities and filtering duplicates...
-> Removed 318 similar poses. Kept 318 highly unique images.


# Yolo filtering to crop out only the cattle

In [ ]:
import os
import cv2
from ultralytics import YOLO
from tqdm import tqdm

def yolo_filter(input_dir, output_dir):
    """
    Scans a directory of raw images, uses YOLOv8 to detect cows (Class 19),
    and saves padded, cropped images to the specified output directory.
    """
    print("Loading YOLOv8 model...")
    model = YOLO('yolov8n.pt')

    COW_CLASS_ID = 19
    os.makedirs(output_dir, exist_ok=True)

    breed_folders = [f for f in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, f)) and not f.startswith('.')]

    for breed in breed_folders:
        print(f"\nCleaning breed: {breed}")
        breed_in_path = os.path.join(input_dir, breed)
        breed_out_path = os.path.join(output_dir, breed)
        os.makedirs(breed_out_path, exist_ok=True)

        image_files = [f for f in os.listdir(breed_in_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
        image_paths = [os.path.join(breed_in_path, f) for f in image_files]

        if not image_paths:
            continue

        # Added conf=0.65 to ignore extreme zooms and bad angles
        results = model.predict(
            source=image_paths, 
            stream=True, 
            verbose=False, 
            classes=[COW_CLASS_ID], 
            device="mps",
            conf=0.65  
        )

        # Zip the original image_paths list alongside the YOLO results generator
        for original_path, res in tqdm(zip(image_paths, results), total=len(image_paths), desc=breed):
            if len(res.boxes) == 0:
                continue

            best_idx = res.boxes.conf.argmax().item()
            best_cow_box = res.boxes.xyxy[best_idx].cpu().numpy()

            img = res.orig_img
            
            x1, y1, x2, y2 = map(int, best_cow_box)

            h, w = img.shape[:2] # image shape is (height, width, channels) 
            pad_x = int((x2 - x1) * 0.05)
            pad_y = int((y2 - y1) * 0.05)

            x1 = max(0, x1 - pad_x)
            y1 = max(0, y1 - pad_y)
            x2 = min(w, x2 + pad_x)
            y2 = min(h, y2 + pad_y)

            cropped_img = img[y1:y2, x1:x2]
            
            out_path = os.path.join(breed_out_path, os.path.basename(original_path))
            cv2.imwrite(out_path, cropped_img)

    print(f"\nAll cropped images successfully saved to {output_dir}")


RAW_DIR = "/Users/satviksingh/Desktop/creating_dataset_cattle/local_raw_frames"
CROPS_DIR = "/Users/satviksingh/Desktop/creating_dataset_cattle/local_cropped_frames"

yolo_filter(input_dir=RAW_DIR, output_dir=CROPS_DIR)

Loading YOLOv8 model...

Cleaning breed: amritmahal


amritmahal: 100%|██████████| 13/13 [00:02<00:00,  5.27it/s]



Cleaning breed: malvi


malvi: 100%|██████████| 10/10 [00:00<00:00, 23.18it/s]



Cleaning breed: bachaur


...


Cleaning breed: red_sindhi


red_sindhi: 100%|██████████| 83/83 [00:16<00:00,  4.91it/s]



Cleaning breed: hariana


hariana: 100%|██████████| 318/318 [02:59<00:00,  1.77it/s]  



All cropped images successfully saved to /Users/satviksingh/Desktop/creating_dataset_cattle/local_cropped_frames


# Passing an another kaggle_dataset through our filters

I found another dataset which contained only 15 breeds but may contain some unique images than we have till now

In [ ]:

RAW_DIR = "/Users/satviksingh/Desktop/creating_dataset_cattle/15_breed"
    
tf_deep_deduplicate(RAW_DIR, threshold=0.82)

Loading TensorFlow MobileNetV2 model...


/var/folders/8j/65h65rn171jc_kw4mq2hh4_w0000gn/T/ipykernel_33549/4165226543.py:21: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')



Extracting features for vechur (716 images)...


100%|██████████| 716/716 [00:54<00:00, 13.19it/s]


Calculating similarities and filtering duplicates...
-> Removed 356 similar poses. Kept 360 highly unique images.

Extracting features for holstein-friesian (975 images)...


...

Calculating similarities and filtering duplicates...
-> Removed 216 similar poses. Kept 340 highly unique images.

Extracting features for gir (870 images)...


100%|██████████| 870/870 [01:03<00:00, 13.73it/s]


Calculating similarities and filtering duplicates...
-> Removed 141 similar poses. Kept 729 highly unique images.

Extracting features for ayrshire (567 images)...


100%|██████████| 567/567 [00:39<00:00, 14.23it/s]


Calculating similarities and filtering duplicates...
-> Removed 172 similar poses. Kept 395 highly unique images.


In [ ]:
RAW_DIR = "/Users/satviksingh/Desktop/creating_dataset_cattle/15_breed"
CROPS_DIR = "/Users/satviksingh/Desktop/creating_dataset_cattle/15_breed_cropped_frames"

yolo_filter(input_dir=RAW_DIR, output_dir=CROPS_DIR)

Loading YOLOv8 model...

Cleaning breed: vechur


vechur: 100%|██████████| 360/360 [00:53<00:00,  6.68it/s] 



Cleaning breed: holstein-friesian


holstein-friesian: 100%|██████████| 732/732 [04:50<00:00,  2.52it/s]  



Cleaning breed: kankrej


...

tharparkar: 100%|██████████| 613/613 [05:19<00:00,  1.92it/s]  



Cleaning breed: umblachery


umblachery: 100%|██████████| 340/340 [00:53<00:00,  6.37it/s] 



Cleaning breed: gir


gir:   0%|          | 0/729 [00:00<?, ?it/s]

# Merging the three datasets and removing duplicates

In [2]:
import os
import hashlib
import shutil

def get_hash(file_path):
    with open(file_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

# 1. DEFINE MASTER PATH
master_path = '/Users/satviksingh/Desktop/creating_dataset_cattle/master_dataset'

Youtube_cattle = '/Users/satviksingh/Desktop/creating_dataset_cattle/youtube_cattle_cropped'
cropped_15_breed = '/Users/satviksingh/Desktop/creating_dataset_cattle/15_breed_cropped_frames'
previous_dataset_cleaned = '/Users/satviksingh/Desktop/creating_dataset_cattle/cattle_dataset_cleaned'

source_folders = [Youtube_cattle, cropped_15_breed, previous_dataset_cleaned]
seen_hashes = set()
duplicates_skipped = 0

for source in source_folders:
    if not os.path.exists(source):
        continue
        
    for breed in os.listdir(source):
        if breed.startswith('.'): 
            continue
            
        src_breed_dir = os.path.join(source, breed)
        if not os.path.isdir(src_breed_dir): 
            continue

        dst_breed_dir = os.path.join(master_path, breed)
        os.makedirs(dst_breed_dir, exist_ok=True)

        for img in os.listdir(src_breed_dir):
            if not img.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
                
            img_path = os.path.join(src_breed_dir, img)
            ext = os.path.splitext(img)[1].lower()  # Preserve original extension
            
            try:
                h = get_hash(img_path)
                if h not in seen_hashes:
                    shutil.copy(img_path, os.path.join(dst_breed_dir, f"{h}{ext}"))
                    seen_hashes.add(h)
                else:
                    duplicates_skipped += 1
            except Exception as e:
                print(f"Error reading {img_path}: {e}")
                continue

print(f"\nMerge complete!")
print(f"Total Unique Images Saved: {len(seen_hashes)}")
print(f"Duplicates Skipped: {duplicates_skipped}")


Merge complete!
Total Unique Images Saved: 11215
Duplicates Skipped: 309


In [ ]:
import os
import shutil

master_path = '/Users/satviksingh/Desktop/creating_dataset_cattle/master_dataset'

if os.path.exists(master_path):
    for breed in os.listdir(master_path):
        if breed.startswith('.'):
            continue
            
        old_dir = os.path.join(master_path, breed)
        if not os.path.isdir(old_dir):
            continue
            
        # Normalizing name: lowercase, strip extra whitespace, replace spaces/hyphens with underscores
        normalized_name = breed.strip().lower().replace(' ', '_').replace('-', '_')
        new_dir = os.path.join(master_path, normalized_name)
        
        # If normalization changes the name or merges folders
        if old_dir != new_dir:
            os.makedirs(new_dir, exist_ok=True)
            for img in os.listdir(old_dir):
                src_img = os.path.join(old_dir, img)
                dst_img = os.path.join(new_dir, img)
                
                # Handle filename collisions if merging into an existing normalized folder
                if os.path.exists(dst_img):
                    base, ext = os.path.splitext(img)
                    counter = 1
                    while os.path.exists(dst_img):
                        dst_img = os.path.join(new_dir, f"{base}_{counter}{ext}")
                        counter += 1
                        
                shutil.move(src_img, dst_img)
            
            # Remove the old unnormalized folder if it's now empty
            try:
                os.rmdir(old_dir)
            except OSError:
                pass

print("Folder name normalization and merge complete!")

Folder name normalization and merge complete!


# Creating the leakproof test and val sets
To augment the dataset, I extracted additional images for each breed from YouTube videos. When constructing the training, validation, and test sets, I implemented a strict video-level splitting strategy. This ensures that all frames originating from a single source video are confined to only one of the sets. This approach is critical to prevent data leakage; although exact duplicates were removed, frames from the same video share highly correlated backgrounds. Without this separation, the model could memorize these background artifacts, leading to artificially inflated performance metrics.


In [ ]:
# 1. Define paths based on your current folder structure
BASE_DIR = "/Users/satviksingh/Desktop/creating_dataset_cattle"
INPUT_VIDEOS_DIR = os.path.join(BASE_DIR, "videos_for_val_test")
TEMP_RAW_DIR = os.path.join(BASE_DIR, "temp_raw_frames")
FINAL_OUTPUT_DIR = os.path.join(BASE_DIR, "frames_from_videos")

# 2. Loop through all breed folders and extract frames
for breed in os.listdir(INPUT_VIDEOS_DIR):
    if breed.startswith('.'): # Ignore hidden files like .DS_Store
        continue
        
    breed_dir = os.path.join(INPUT_VIDEOS_DIR, breed)
    if not os.path.isdir(breed_dir):
        continue
        
    # Create an output folder for this breed's raw frames
    breed_raw_output = os.path.join(TEMP_RAW_DIR, breed)
    os.makedirs(breed_raw_output, exist_ok=True)

    # Process each video in the breed folder
    for video_file in os.listdir(breed_dir):
        if not video_file.endswith(('.mp4', '.mkv', '.webm')):
            continue
            
        video_path = os.path.join(breed_dir, video_file)
        
        # Call your existing extraction function
        extract_frames(
            video_path=video_path, 
            output_folder=breed_raw_output, 
            frame_skip=60, 
            prefix=f"{breed}_{video_file.split('.')[0]}"
        )

# 3. Run TensorFlow deduplication on the extracted frames
print("\n--- Starting Deduplication ---")
tf_deep_deduplicate(raw_dir=TEMP_RAW_DIR, threshold=0.82)

# 4. Run YOLO filtering to crop out the cattle and save to the final folder
print("\n--- Starting YOLO Cropping ---")
yolo_filter(input_dir=TEMP_RAW_DIR, output_dir=FINAL_OUTPUT_DIR)

print(f"\nPipeline complete! Your final processed images are saved in {FINAL_OUTPUT_DIR}")

In [ ]:
import os
import json

output_root = '/Users/satviksingh/Desktop/creating_dataset_cattle/master_dataset_capped'

train_dir = os.path.join(output_root, "train")

class_names = sorted([
    folder
    for folder in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, folder))
    and not folder.startswith(".")
])

# Check that there are exactly 50 classes
assert len(class_names) == 50, f"Expected 50 classes, found {len(class_names)}"

# Check that validation and test contain exactly the same class names
val_dir = os.path.join(output_root, "val")
test_dir = os.path.join(output_root, "test")

val_classes = sorted([
    folder
    for folder in os.listdir(val_dir)
    if os.path.isdir(os.path.join(val_dir, folder))
    and not folder.startswith(".")
])

test_classes = sorted([
    folder
    for folder in os.listdir(test_dir)
    if os.path.isdir(os.path.join(test_dir, folder))
    and not folder.startswith(".")
])

assert class_names == val_classes, "Train and val class names do not match!"
assert class_names == test_classes, "Train and test class names do not match!"

# Save class_names.json alongside train/val/test
json_path = os.path.join(output_root, "class_names.json")

with open(json_path, "w") as fp:
    json.dump(class_names, fp, indent=2)

print(f"Saved class_names.json to: {json_path}")
print(f"Number of classes: {len(class_names)}")
print("\nClass order:")
for i, name in enumerate(class_names):
    print(f"{i}: {name}")

Saved class_names.json to: /Users/satviksingh/Desktop/creating_dataset_cattle/master_dataset_capped/class_names.json
Number of classes: 50

Class order:
0: Amritmahal
1: Ayrshire
2: Bachaur
3: Badri
4: Bargur
5: Bhelai
6: Dagri
7: Dangi
8: Deoni
9: Gangatari
10: Gaolao
11: Ghumsari
12: Gir
13: Hallikar
14: Hariana
15: Himachali_pahari
16: Kangayam
17: Kankrej
18: Kenkatha
19: Khariar
20: Kherigarh
21: Khillari
22: Konkan_kapila
23: Kosali
24: Krishna_valley
25: Ladakhi
26: Lakhimi
27: Malnad_gidda
28: Malvi
29: Mewati
30: Motu
31: Nagori
32: Nari
33: Nimari
34: Ongole
35: Poda_thirupu
36: Ponwar
37: Pulikulam
38: Punganur
39: Purnea
40: Rathi
41: Red_kandhari
42: Red_sindhi
43: Sahiwal
44: Shweta_Kapila
45: Siri
46: Tharparkar
47: Thutho
48: Umblachery
49: Vechur
